# A4 · Trust your model: error bars from one sample (the central limit theorem and the bootstrap)
**Series A: Machine Learning Basics for Chemists, Post 4 of 7** &nbsp;|&nbsp; companion *primer* notebook

Section 2.3.4 of the notes asks for the error of a result that you can compute only once. The exercise (the `dice` notebook, *The Central Limit Theorem, and how the bootstrap puts it to work*) answers with a fair die, for which the truth is known: the spread of an average shrinks as $\sigma/\sqrt N$, its shape is a Gaussian, and the **bootstrap** measures that spread from a single sample. This primer rebuilds the whole exercise **with NumPy only** (no SciPy), by making the *same random-number calls in the same order* as the notebook, so that **every number it printed is reproduced to the last digit**; then it answers what the notebook leaves open (the small-$N$ coverage, a skewed population, the bias of the bootstrap for a median, the maximum, the length of the blocks) and applies the bootstrap to the question of the previous post: *how much can a test AUC of a handful of molecules be trusted?* Part II checks the functions against SciPy, including `scipy.stats.bootstrap` with its BCa interval.

> **Two parts.** **Part I** needs only NumPy, pandas and matplotlib and runs in well under a minute on a CPU (the reproduction of the notebook has to make all of its random draws). **Part II** needs SciPy (preinstalled in Colab).
>
> **Reading the code.** Every section starts with a short **Stop and predict** question: answer it before running the cell.

| Section | What you do |
|---|---|
| 1 | one experiment: 100 rolls of a die |
| 2 | the sampling distribution of the mean, and a continuity correction |
| 3 | the $1/\sqrt N$ law |
| 4 | the shape: an exponential population; no variance: the Cauchy |
| 5 | the bootstrap: one sample, and what $B$ buys |
| 6 | does the interval cover? the median; dependent data |
| 7 | beyond the notebook: small $N$, skewed data, bias, the maximum, block lengths |
| 8 | a test AUC of 19 molecules: bootstrap error bars for a model |
| **Part II** | cross-check with SciPy |

## 0. Setup and helper functions

In [ ]:
import math, time, importlib.util
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True, linewidth=120)
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (6.5, 3.6), "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")
# the few statistical functions the notebook takes from SciPy, written out
norm_cdf = lambda z: 0.5 * (1 + np.vectorize(math.erf)(np.asarray(z, float) / math.sqrt(2)))
skewness = lambda x: ((x - x.mean()) ** 3).mean() / x.std() ** 3                       # the biased moment estimates (SciPy's defaults)
ex_kurtosis = lambda x: ((x - x.mean()) ** 4).mean() / x.std() ** 4 - 3
iqr = lambda x: np.percentile(x, 75) - np.percentile(x, 25)
def t975(nu):
    """the 97.5 % quantile of Student's t with nu degrees of freedom (Cornish-Fisher expansion in 1/nu)"""
    z = 1.959963984540054; return z + (z ** 3 + z) / (4 * nu) + (5 * z ** 5 + 16 * z ** 3 + 3 * z) / (96 * nu ** 2) + (3 * z ** 7 + 19 * z ** 5 + 17 * z ** 3 - 15 * z) / (384 * nu ** 3)
def bootstrap_means(x, n_boot, rng, chunk=2000):
    """means of n_boot resamples of x, each the same size as x, drawn WITH replacement from x itself"""
    x = np.asarray(x); out = np.empty(n_boot)
    for a in range(0, n_boot, chunk):
        b = min(a + chunk, n_boot); idx = rng.integers(0, len(x), size=(b - a, len(x))); out[a:b] = x[idx].mean(axis=1)
    return out
print("Student t, 97.5 %: 99 degrees of freedom", round(t975(99), 5), "| 9 degrees of freedom", round(t975(9), 5), "(exact: 1.98422 and 2.26216)")
assert abs(t975(99) - 1.98422) < 1e-4 and abs(t975(9) - 2.26216) < 1e-3 and abs(norm_cdf(1.0) - 0.8413447) < 1e-6

## 1. One experiment: 100 rolls of a fair die
A fair die has the exact population mean $\mu=3.5$, variance $\sigma^2=35/12$ and $\sigma=1.7078$. The **standard error** of the mean of $N$ independent rolls is $\epsilon=\sigma/\sqrt N$: the spread of the *average* over repeated experiments, not of one roll. We use one random-number generator for everything, with the seed 2024, and the three sizes of the exercise: $N=100$ rolls per experiment, $M=10{,}000$ repetitions, $B=10{,}000$ bootstrap resamples.
**Stop and predict:** the standard error for $N=100$, and how far from the expected count $N/6$ a face can be before it looks unusual.

In [ ]:
rng = np.random.default_rng(2024); N, M, B = 100, 10_000, 10_000
faces = np.arange(1, 7); mu_true = faces.mean(); var_true = faces.var(); sigma_true = math.sqrt(var_true); se_theory = sigma_true / math.sqrt(N)
print(f"population: mean {mu_true}, variance {var_true:.4f} (= 35/12), sigma {sigma_true:.4f} | standard error of the mean of N = {N} rolls: {se_theory:.4f}")
rolls = rng.integers(1, 7, size=N); counts = np.bincount(rolls, minlength=7)[1:]
print("counts of 1..6:", counts, "(sum", counts.sum(), ") | sample mean %.3f | sample sigma ddof=0: %.3f, ddof=1: %.3f" % (rolls.mean(), rolls.std(ddof=0), rolls.std(ddof=1)))
sd_count = math.sqrt(N * (1 / 6) * (5 / 6)); z = (counts - N / 6) / sd_count
print(f"each count is binomial: mean {N / 6:.1f}, s.d. {sd_count:.2f}; the counts are {np.round(z, 2)} standard deviations from it | the sample mean is {(rolls.mean() - mu_true) / se_theory:+.2f} standard errors from 3.5")
assert counts.tolist() == [18, 24, 19, 9, 15, 15] and abs(rolls.mean() - 3.24) < 1e-9 and abs(rolls.std() - 1.715) < 5e-4 and abs(z[1] - 1.97) < 0.01 and abs(z[3] + 2.06) < 0.01

## 2. The sampling distribution of the mean
Repeat the experiment $M=10{,}000$ times and keep each mean: the histogram is the **sampling distribution of the mean**, what you would have got by running again. Theory says it is centred on $3.5$ with standard deviation $\epsilon=0.1708$ and Gaussian in shape. One subtlety: a mean of 100 integers is a multiple of $0.01$, so a fraction "within one standard error" must be compared with the Gaussian using the **midpoint** between grid points as the cut-off (a continuity correction).
**Stop and predict:** the fractions within 1, 2 and 3 standard errors; will they equal $68.27\%$, $95.45\%$, $99.73\%$ exactly?

In [ ]:
means = rng.integers(1, 7, size=(M, N)).mean(axis=1)
print(f"mean of the {M} means {means.mean():.4f} (theory 3.5) | s.d. of the means {means.std():.4f} (theory {se_theory:.4f})")
fr, fr_cc, gauss = [], [], []
for k in (1, 2, 3):
    frac = np.mean(np.abs(means - mu_true) < k * se_theory); cut = (math.floor(k * se_theory / 0.01 + 1e-9) + 0.5) * 0.01              # midpoint between the two grid points around k standard errors
    fr.append(frac); gauss.append(float(norm_cdf(k) - norm_cdf(-k))); fr_cc.append(float(norm_cdf(cut / se_theory) - norm_cdf(-cut / se_theory)))
    print(f"within {k} standard errors: observed {frac:.4f} | Gaussian {gauss[-1]:.4f} | Gaussian with the continuity correction (cut-off {cut:.3f}) {fr_cc[-1]:.4f}")
assert abs(means.mean() - 3.4989) < 5e-5 and abs(means.std() - 0.1717) < 5e-5 and np.allclose(fr, [0.6941, 0.9547, 0.9979]) and np.allclose(fr_cc, [0.6945, 0.9566, 0.9974], atol=1e-4)
fig, ax = plt.subplots(); edges = np.arange(means.min() - 0.005, means.max() + 0.015, 0.02); ax.hist(means, bins=edges, density=True, color=COLORS["blue"], alpha=0.7, label=f"{M} means of N = {N} rolls")
xs = np.linspace(means.min(), means.max(), 400); ax.plot(xs, np.exp(-0.5 * ((xs - mu_true) / se_theory) ** 2) / (se_theory * math.sqrt(2 * math.pi)), "k", lw=1.5, label=r"Gaussian, $\mu=3.5$, $\epsilon=\sigma/\sqrt{N}$")
ax.set(xlabel="mean of 100 rolls", ylabel="probability density", title="the distribution of the mean is a Gaussian"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## 3. The $1/\sqrt N$ law
Because $\epsilon=\sigma/\sqrt N$, halving the error costs four times the data and one more decimal digit a hundred times. On a log-log plot the law is a straight line of slope $-1/2$.
**Stop and predict:** how many rolls give a standard error of $0.05$, and of $0.017$?

In [ ]:
Ns = np.array([1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]); reps = 4000
se_obs = np.array([rng.integers(1, 7, size=(reps, n)).mean(axis=1).std() for n in Ns])
tab = pd.DataFrame({"N": Ns, "observed s.d. of the mean": se_obs, "theory sigma/sqrt(N)": sigma_true / np.sqrt(Ns)}); print(tab.round(4).to_string(index=False))
slope = np.polyfit(np.log(Ns), np.log(se_obs), 1)[0]; print(f"\nfitted log-log slope {slope:.3f} (theory -0.5) | rolls for a standard error of 0.05: {(sigma_true / 0.05) ** 2:.0f}; of 0.017: {(sigma_true / 0.017) ** 2:.0f}")
assert abs(slope + 0.503) < 5e-4 and abs(se_obs[6] - 0.1706) < 5e-5 and abs(se_obs[0] - 1.7041) < 5e-5

## 4. The shape: a skewed population, and a population without a variance
The Gaussian shape does not need a symmetric population: for the **exponential** distribution (skewness $2$, excess kurtosis $6$) the skewness of the mean of $N$ draws is $2/\sqrt N$ and its excess kurtosis $6/N$, so the shape converges gradually. The **Cauchy** distribution has no variance and no mean; the average of $N$ independent Cauchy numbers is distributed like a single one, so averaging does nothing, and the spread must be measured by the interquartile range (IQR), exactly $2$ for one standard Cauchy.
**Stop and predict:** at which $N$ does the exponential's mean look Gaussian, and what does the IQR of a Cauchy mean do as $N$ grows 1000-fold?

In [ ]:
Msk = 20_000; rows = []; sk_means = {}
for n in (1, 2, 5, 30, 100):
    m = rng.exponential(1.0, size=(Msk, n)).mean(axis=1); sk_means[n] = m; rows.append((n, skewness(m), 2 / math.sqrt(n), ex_kurtosis(m), 6 / n))
print(pd.DataFrame(rows, columns=["N", "skewness of the mean", "theory 2/sqrt(N)", "excess kurtosis", "theory 6/N"]).round(3).to_string(index=False))
T = 100_000; die_run = np.cumsum(rng.integers(1, 7, size=T)) / np.arange(1, T + 1); cau_run = np.cumsum(rng.standard_cauchy(size=T)) / np.arange(1, T + 1); rows = []
for n in (1, 10, 100, 1000): rows.append((n, iqr(rng.standard_cauchy(size=(2000, n)).mean(axis=1)), iqr(rng.integers(1, 7, size=(2000, n)).mean(axis=1))))
print("\n" + pd.DataFrame(rows, columns=["N", "IQR of the mean (Cauchy)", "IQR of the mean (die)"]).round(3).to_string(index=False))
print(f"running mean after {T} samples: die {die_run[-1]:.3f}, Cauchy {cau_run[-1]:.3f}; largest jump of the Cauchy running mean after t = 1000: {np.abs(np.diff(cau_run[1000:])).max():.3f}")
assert abs(rows[0][1] - 1.982) < 5e-4 and abs(rows[3][2] - 0.074) < 5e-4
fig, ax = plt.subplots(); ax.semilogx(np.arange(1, T + 1), die_run, color=COLORS["blue"], label="fair die"); ax.semilogx(np.arange(1, T + 1), cau_run, color=COLORS["orange"], label="Cauchy"); ax.axhline(3.5, color="k", lw=0.6)
ax.set(xlabel="number of samples t", ylabel="running mean", title="the running mean converges for the die, not for the Cauchy"); ax.legend(); plt.tight_layout(); plt.show()

## 5. The bootstrap: one sample, and what $B$ buys
In practice there is one sample and the population is unknown. The **bootstrap** treats the sample as the population: draw $N$ values *from the sample, with replacement*, compute the statistic, repeat $B$ times; the spread of the results is the **bootstrap standard error**, and the 2.5th and 97.5th percentiles give a 95% **percentile interval**. The bootstrap distribution is centred on the *sample's* statistic, not on the truth: it measures how much an estimate fluctuates, not how far this particular estimate is from the truth. The number of resamples $B$ only controls the simulation noise.
**Stop and predict:** where will the bootstrap histogram be centred, and what will it do to the answer if $B$ grows from $10$ to $10^5$?

In [ ]:
sample = rng.integers(1, 7, size=N); xbar, s0 = sample.mean(), sample.std(ddof=0); boot = bootstrap_means(sample, B, rng)
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"the one sample: mean {xbar:.3f}, sigma(ddof=0) {s0:.3f} | plug-in prediction s0/sqrt(N) = {s0 / math.sqrt(N):.4f}")
print(f"bootstrap: mean of the boot means {boot.mean():.3f}, standard error {boot.std():.4f} | truth (known here): centre 3.5, s.d. {se_theory:.4f}")
print(f"95% percentile interval [{lo:.3f}, {hi:.3f}] contains 3.5: {lo <= 3.5 <= hi}")
assert abs(xbar - 3.42) < 1e-9 and abs(boot.std() - 0.1745) < 5e-5 and abs(lo - 3.08) < 1e-9 and abs(hi - 3.76) < 1e-9
rows = []
for Bk in (10, 100, 1000, 10_000, 100_000):
    ses = [bootstrap_means(sample, Bk, rng).std() for _ in range(30)]; rows.append((Bk, np.mean(ses), np.std(ses)))
bt = pd.DataFrame(rows, columns=["B", "mean of 30 bootstrap SEs", "spread (s.d.) over the 30"]); print("\n" + bt.round(4).to_string(index=False))
print(f"plug-in value {s0 / math.sqrt(N):.4f}; true standard error {se_theory:.4f}: the bootstrap converges to the first, and no resampling can reach the second")
assert abs(bt.iloc[4, 1] - 0.1739) < 5e-5 and abs(bt.iloc[0, 2] - 0.0402) < 5e-5
fig, ax = plt.subplots(); edges = np.arange(boot.min() - 0.005, boot.max() + 0.015, 0.02); ax.hist(boot, bins=edges, density=True, color=COLORS["green"], alpha=0.7, label="bootstrap means (from ONE sample)")
xs = np.linspace(boot.min(), boot.max(), 400); ax.plot(xs, np.exp(-0.5 * ((xs - 3.5) / se_theory) ** 2) / (se_theory * math.sqrt(2 * math.pi)), "k", lw=1.5, label="true sampling distribution (needs the population)")
ax.axvline(xbar, color=COLORS["green"], ls="--", label="sample mean"); ax.axvline(3.5, color="k", ls=":", label="true mean"); ax.set(xlabel="mean", ylabel="density", title="bootstrap: same width, centred on the sample mean")
ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2); plt.tight_layout(); plt.show()

## 6. Does the interval cover? The median; dependent data
A 95% interval is a promise about a *procedure*: repeated on new samples, about 95% of the intervals should contain the truth (the **coverage**). The bootstrap is not limited to means: replace "mean" by "median" and nothing else changes. And everything so far assumed that the values are **independent**; for a correlated series (an AR(1) process, $x_t=\phi\,x_{t-1}+\text{noise}$) only $N_{\rm eff}=N/g$ values are independent, $g=(1+\phi)/(1-\phi)$, and resampling single points hides the correlation; resampling **blocks** of consecutive points keeps it.
**Stop and predict:** the coverage of the percentile interval for $N=100$ rolls, and how much too small the ordinary bootstrap error bar of an AR(1) series with $\phi=0.9$ will be.

In [ ]:
reps = 1000; hit_b = hit_t = 0; widths = []
for _ in range(reps):
    x = rng.integers(1, 7, size=N); bm = bootstrap_means(x, 1000, rng); l_, h_ = np.percentile(bm, [2.5, 97.5]); hit_b += (l_ <= 3.5 <= h_); widths.append(h_ - l_)
    half = t975(N - 1) * x.std(ddof=1) / math.sqrt(N); hit_t += (x.mean() - half <= 3.5 <= x.mean() + half)
print(f"coverage (target 0.950 +- {math.sqrt(.95 * .05 / reps):.3f}): bootstrap percentile {hit_b / reps:.3f}, classical t {hit_t / reps:.3f} | mean width {np.mean(widths):.3f} (ideal {2 * 1.96 * se_theory:.3f})")
cover_b, cover_t = hit_b / reps, hit_t / reps
assert cover_b == 0.941 and cover_t == 0.941 and abs(np.mean(widths) - 0.662) < 5e-4
true_median = math.log(2); se_asym = 1 / math.sqrt(N); meds = np.median(rng.exponential(1.0, size=(20_000, N)), axis=1); boot_se = []
for _ in range(200):
    x = rng.exponential(1.0, size=N); idx = rng.integers(0, N, size=(2000, N)); boot_se.append(np.median(x[idx], axis=1).std())
boot_se = np.array(boot_se); print(f"median of an exponential sample: true standard error {meds.std():.4f} (asymptotic formula {se_asym:.4f}); bootstrap SE from a single sample over 200 samples: mean {boot_se.mean():.4f}, spread {boot_se.std():.4f}")
assert abs(meds.std() - 0.0999) < 5e-5 and abs(boot_se.mean() - 0.1040) < 5e-5 and abs(boot_se.std() - 0.0257) < 5e-5

In [ ]:
phi, Nt, Mt = 0.9, 2000, 2000; var_x = 1 / (1 - phi ** 2); g = (1 + phi) / (1 - phi)
def ar1_series(n_series, n, phi, rng):
    x = np.empty((n_series, n)); x[:, 0] = rng.normal(0, math.sqrt(1.0 / (1 - phi ** 2)), n_series); eps = rng.normal(size=(n_series, n))
    for t in range(1, n): x[:, t] = phi * x[:, t - 1] + eps[:, t]
    return x
print(f"statistical inefficiency g = {g:.1f}: N_eff = N/g = {Nt / g:.0f} of {Nt}")
true_se = ar1_series(Mt, Nt, phi, rng).mean(axis=1).std(); print(f"true s.d. of the mean over {Mt} series {true_se:.4f} (theory {math.sqrt(var_x * g / Nt):.4f}) | naive s/sqrt(N) {math.sqrt(var_x / Nt):.4f}: too small by sqrt(g) = {math.sqrt(g):.2f}")
x = ar1_series(1, Nt, phi, rng)[0]; naive = bootstrap_means(x, 4000, rng).std(); L = 50; starts = rng.integers(0, Nt, size=(4000, Nt // L)); idx = (starts[:, :, None] + np.arange(L)).reshape(4000, -1) % Nt; block = x[idx].mean(axis=1).std()
print(f"one series: ordinary bootstrap SE {naive:.4f}, block bootstrap SE (L = {L}) {block:.4f}")
assert abs(true_se - 0.2199) < 5e-5 and abs(naive - 0.0497) < 5e-5 and abs(block - 0.1985) < 5e-5
print("\nAll the numbers printed by the notebook are reproduced to the last digit.")

## 7. Beyond the notebook
The notebook leaves several questions to the exercises or marks them as not tested. From here on we use a *new* generator (seed 7), so the reproduction above is not disturbed.
**(a) Small $N$.** Does the percentile interval reach 95%? **(b)** And for a *skewed* population (the exponential)? **(c)** The notebook finds that the bootstrap SE of the median averages $0.1040$ against a true $0.0999$ and adds "a small positive bias is possible and I have not tested for it": we test it. **(d)** The bootstrap of the *maximum* of a uniform sample fails, in what way? **(e)** How long must the blocks be?
**Stop and predict:** the coverage of the bootstrap interval at $N=10$ for the die, and for the exponential.

In [ ]:
rng2 = np.random.default_rng(7); t0 = time.perf_counter(); rows = []
for pop, draw, truth in (("fair die", lambda n: rng2.integers(1, 7, size=n), 3.5), ("exponential", lambda n: rng2.exponential(1.0, size=n), 1.0)):
    for n, reps in ((10, 3000), (30, 3000), (100, 1500)):
        hb = ht = 0
        for _ in range(reps):
            x = draw(n); bm = bootstrap_means(x, 1000, rng2); l_, h_ = np.percentile(bm, [2.5, 97.5]); hb += (l_ <= truth <= h_); hh = t975(n - 1) * x.std(ddof=1) / math.sqrt(n); ht += (x.mean() - hh <= truth <= x.mean() + hh)
        rows.append((pop, n, reps, hb / reps, ht / reps))
cov = pd.DataFrame(rows, columns=["population", "N", "trials", "bootstrap percentile", "classical t"]); print(cov.round(3).to_string(index=False)); print(f"[{time.perf_counter() - t0:.0f} s]")
assert cov.iloc[0, 3] < 0.94 and cov.iloc[1, 3] > cov.iloc[0, 3] and cov.iloc[3, 3] < cov.iloc[0, 3] and cov.iloc[5, 3] < 0.945

In [ ]:
t0 = time.perf_counter(); true_se_med = np.median(rng2.exponential(1.0, size=(40_000, 100)), axis=1).std(); ses = []
for _ in range(1500):
    x = rng2.exponential(1.0, size=100); idx = rng2.integers(0, 100, size=(1000, 100)); ses.append(np.median(x[idx], axis=1).std())
ses = np.array(ses); se_of_mean = ses.std() / math.sqrt(len(ses)); rel = ses.mean() / true_se_med - 1
print(f"(c) median: true SE {true_se_med:.4f}; mean bootstrap SE over 1500 samples {ses.mean():.4f} +- {se_of_mean:.4f}: bias {100 * rel:+.1f} % (about {(ses.mean() - true_se_med) / math.hypot(se_of_mean, true_se_med / math.sqrt(2 * 40_000)):.1f} standard errors); spread of a single bootstrap SE {ses.std():.4f} ({100 * ses.std() / ses.mean():.0f} % of its value)")
fr_eq, se_max = [], []
for _ in range(300):
    x = rng2.uniform(size=100); m = x[rng2.integers(0, 100, size=(2000, 100))].max(axis=1); fr_eq.append((m == x.max()).mean()); se_max.append(m.std())
true_max_se = math.sqrt(100 / (101 ** 2 * 102))
print(f"(d) maximum of 100 uniform numbers: {np.mean(fr_eq):.3f} of the resamples have a maximum EQUAL to the sample maximum (theory 1 - (1 - 1/N)^N = {1 - (1 - 1 / 100) ** 100:.3f}); the true sampling distribution is continuous; yet the bootstrap SE {np.mean(se_max):.4f} is near the true {true_max_se:.4f} (ratio {np.mean(se_max) / true_max_se:.2f}); the upper end of its percentile interval is the sample maximum itself, so it can never contain the population maximum 1")
print(f"    exercise 7: sqrt((N-1)/N) = {math.sqrt(99 / 100):.3f} at N = 100 and {math.sqrt(9 / 10):.3f} at N = 10 | exercise 8: g = 40, N = 80000: N_eff = {80000 // 40}, the standard error is larger than the naive one by sqrt(g) = {math.sqrt(40):.1f} [{time.perf_counter() - t0:.0f} s]")
assert 0.01 < rel < 0.04 and abs(np.mean(fr_eq) - 0.634) < 0.01

In [ ]:
def block_se(x, L, B_, rg):
    n = len(x); starts = rg.integers(0, n, size=(B_, n // L)); idx = (starts[:, :, None] + np.arange(L)).reshape(B_, -1) % n; return x[idx].mean(axis=1).std()
t0 = time.perf_counter(); res = {}
for ph in (0.5, 0.9):
    n = 2000; truth = ar1_series(2000, n, ph, rng2).mean(axis=1).std(); Xs = ar1_series(30, n, ph, rng2); res[ph] = (truth, [(L_, np.mean([block_se(x_, L_, 1000, rng2) for x_ in Xs])) for L_ in (1, 5, 10, 20, 50, 100, 200, 400)])
for ph in (0.5, 0.9):
    truth, lst = res[ph]; print(f"(e) phi = {ph}: g = {(1 + ph) / (1 - ph):.0f}, true SE {truth:.4f}, naive {math.sqrt(1 / (1 - ph ** 2) / 2000):.4f}\n    block length L:      " + "  ".join(f"{L_:5d}" for L_, _ in lst) + "\n    SE / truth:          " + "  ".join(f"{v / truth:5.2f}" for _, v in lst))
print(f"[{time.perf_counter() - t0:.0f} s]")
best9 = max(v for _, v in res[0.9][1]) / res[0.9][0]; assert best9 < 0.95 and res[0.5][1][0][1] / res[0.5][0] < 0.7
fig, ax = plt.subplots(figsize=(6.6, 3.6))
for ph, c_ in ((0.5, COLORS["blue"]), (0.9, COLORS["red"])): truth, lst = res[ph]; ax.semilogx([L_ for L_, _ in lst], [v / truth for _, v in lst], "o-", color=c_, label=fr"$\phi$ = {ph} (g = {(1 + ph) / (1 - ph):.0f})")
ax.axhline(1, color="k", ls="--", lw=1); ax.set(xlabel="block length L", ylabel="block bootstrap SE / true SE", title="block bootstrap of an AR(1) series (N = 2000)"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## 8. A test AUC of 19 molecules: bootstrap error bars for a model
The previous post (A6b) left a test set of $19$ molecules with $4$ positives (lipophilic) and one of $16$ molecules with $2$ positives (hbond) and the AUC $0.833$ and $0.857$ of a logistic regression. The bootstrap gives such a number an error bar: resample the *molecules* (score and label together) and recompute the AUC. Two problems appear: a resample may contain no positive (the AUC is undefined), and the interval is capped at $1$. A **stratified** bootstrap resamples positives and negatives separately and avoids the first.
**Stop and predict:** the fraction of resamples without a positive molecule for $4$ positives among $19$ and for $2$ among $16$.

In [ ]:
LIP_S, LIP_Y = np.array([-1.645, -2.3523, 0.9312, -2.2633, -1.2434, -1.7722, -1.0445, -0.6788, 0.9619, -2.0774, 0.1331, -2.1263, -1.3215, -0.6454, -2.7481, 0.5467, -1.1068, -1.4096, -0.4553]), np.array([0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0])
HB_S, HB_Y = np.array([-2.3246, -1.3425, -2.9397, -2.2522, -3.0662, -1.8629, -2.8618, -2.4486, -2.9102, -2.1999, -2.7539, -3.1248, -0.2668, -3.1836, -1.9324, -3.101]), np.array([0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0])

In [ ]:
def auc(y, s):
    pos, neg = s[y == 1], s[y == 0]; return ((pos[:, None] > neg[None, :]).sum() + 0.5 * (pos[:, None] == neg[None, :]).sum()) / (len(pos) * len(neg))
rg = np.random.default_rng(11); out = []
for name, s, y in (("lipophilic", LIP_S, LIP_Y), ("hbond", HB_S, HB_Y)):
    n = len(y); aucs, undefined = [], 0
    for _ in range(10_000):
        i = rg.integers(0, n, n); yy = y[i]
        if yy.min() == yy.max(): undefined += 1; continue
        aucs.append(auc(yy, s[i]))
    aucs = np.array(aucs); pos, neg = np.where(y == 1)[0], np.where(y == 0)[0]; st = []
    for _ in range(10_000): i = np.r_[rg.choice(pos, len(pos)), rg.choice(neg, len(neg))]; st.append(auc(y[i], s[i]))
    st = np.array(st); p0 = (1 - y.mean()) ** n
    out.append((name, n, int(y.sum()), auc(y, s), 100 * undefined / 10_000, 100 * p0, aucs.std(), *np.percentile(aucs, [2.5, 97.5]), st.std(), *np.percentile(st, [2.5, 97.5])))
tb = pd.DataFrame(out, columns=["task", "n", "positives", "AUC", "undefined (%)", "theory (%)", "iid SE", "iid 2.5%", "iid 97.5%", "stratified SE", "str. 2.5%", "str. 97.5%"]); print(tb.round(3).to_string(index=False))
print("\nthe notebook of post A6b found, over 200 random splits, a spread of 0.083 (lipophilic) and 0.134 (hbond) for the test AUC: the bootstrap error bar of a single test set is of the same size")
assert abs(tb.iloc[0, 3] - 0.8333) < 1e-3 and abs(tb.iloc[1, 4] - tb.iloc[1, 5]) < 1.0 and tb.iloc[0, 7] < 0.7 and tb.iloc[1, 7] < 0.7 and tb.iloc[:, 8].max() == 1.0
fig, ax = plt.subplots(figsize=(6.6, 3.6)); a_ = []
for _ in range(10_000):
    i = rg.integers(0, len(LIP_Y), len(LIP_Y)); yy = LIP_Y[i]
    if yy.min() != yy.max(): a_.append(auc(yy, LIP_S[i]))
ax.hist(a_, bins=np.linspace(0.2, 1.0, 33), color=COLORS["blue"], alpha=0.8); ax.axvline(auc(LIP_Y, LIP_S), color=COLORS["red"], ls="--", label="AUC of the test set (0.833)")
ax.set(xlabel="AUC of a bootstrap resample", ylabel="resamples of 10,000", title="19 molecules, 4 positives: how much is the AUC worth?"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## 9. Your turn
**Exercise 1.** Repeat Section 6 for the median of a die ($N=100$) and check the bootstrap interval's coverage.
**Exercise 2.** Use a skewed population with skewness 10 (a log-normal) and find the $N$ at which the percentile interval covers $95\%$.
**Exercise 3.** Replace the percentile interval by the *basic* (pivotal) interval $2\hat\theta-q_{97.5},\,2\hat\theta-q_{2.5}$ and compare the coverage for the exponential at $N=30$.
**Exercise 4.** For the AUC of Section 8, use $B=100{,}000$. Does the interval change in the second digit?
**Exercise 5.** Thin the AR(1) series by $g$ (keep every 19th point) and use the ordinary bootstrap. Compare with the block bootstrap.

## 10. Self-check

In [ ]:
assert counts.tolist() == [18, 24, 19, 9, 15, 15] and abs(means.std() - 0.1717) < 5e-5 and abs(slope + 0.503) < 5e-4 and abs(boot.std() - 0.1745) < 5e-5
assert cover_b == 0.941 and abs(boot_se.mean() - 0.1040) < 5e-5 and abs(block - 0.1985) < 5e-5 and 0.01 < rel < 0.04 and tb.iloc[:, 8].max() == 1.0
print("All checks passed: you reproduced the numbers in the post.")

# Part II. Cross-check with SciPy
> **Read this first.** Part II compares the helper functions of Part I with SciPy (preinstalled in Colab): the skewness and kurtosis, the interquartile range, the normal distribution function and Student's $t$ quantile, and then the bootstrap itself against `scipy.stats.bootstrap` (percentile and BCa intervals) for the one sample of Section 5 and for the median. The random draws of SciPy differ from ours, so these comparisons use a tolerance that reflects the simulation noise of $B$ resamples. Each stage is guarded; the last cell lists the checks that passed.

In [ ]:
HAVE_SP = importlib.util.find_spec("scipy") is not None; checks = []
def check(label, ok): checks.append((label, bool(ok))); print(("   ok   " if ok else "  FAIL  ") + label)
if HAVE_SP:
    import scipy, warnings; warnings.filterwarnings("ignore"); from scipy import stats; print("scipy", scipy.__version__)
    try:
        m5 = sk_means[5]; check("skewness and excess kurtosis equal scipy.stats.skew and kurtosis", abs(skewness(m5) - stats.skew(m5)) < 1e-12 and abs(ex_kurtosis(m5) - stats.kurtosis(m5)) < 1e-12)
        check("the interquartile range equals scipy.stats.iqr", abs(iqr(means) - stats.iqr(means)) < 1e-12)
        zz = np.linspace(-4, 4, 17); check("the normal distribution function (math.erf) equals scipy.stats.norm.cdf", np.abs(norm_cdf(zz) - stats.norm.cdf(zz)).max() < 1e-12)
        check(f"Student's t quantiles: {t975(99):.5f} and {t975(9):.5f} against SciPy's {stats.t.ppf(0.975, 99):.5f} and {stats.t.ppf(0.975, 9):.5f}", abs(t975(99) - stats.t.ppf(0.975, 99)) < 1e-4 and abs(t975(9) - stats.t.ppf(0.975, 9)) < 1e-3)
        r_ = stats.bootstrap((sample,), np.mean, n_resamples=9999, method="percentile", random_state=np.random.default_rng(5)); lo_s, hi_s = r_.confidence_interval
        check(f"scipy.stats.bootstrap, one sample of the die: standard error {r_.standard_error:.4f} against ours {boot.std():.4f}; interval [{lo_s:.3f}, {hi_s:.3f}] against [{lo:.3f}, {hi:.3f}]", abs(r_.standard_error - boot.std()) < 0.003 and abs(lo_s - lo) < 0.03 and abs(hi_s - hi) < 0.03)
        r_b = stats.bootstrap((sample,), np.mean, n_resamples=9999, method="BCa", random_state=np.random.default_rng(5)); print(f"   BCa interval of the same mean: [{r_b.confidence_interval.low:.3f}, {r_b.confidence_interval.high:.3f}] (percentile: [{lo_s:.3f}, {hi_s:.3f}])")
        xm = rng2.exponential(1.0, size=100); r_m = stats.bootstrap((xm,), np.median, n_resamples=2000, method="percentile", random_state=np.random.default_rng(6)); idx = rng2.integers(0, 100, size=(2000, 100)); ours = np.median(xm[idx], axis=1).std()
        check(f"the bootstrap SE of a median: scipy {r_m.standard_error:.4f}, ours {ours:.4f} (agree within the simulation noise)", abs(r_m.standard_error - ours) < 0.015)
    except Exception as e: print("stage could not be completed:", type(e).__name__, e)
    ok = [lab for lab, p in checks if p]; bad = [lab for lab, p in checks if not p]; print(f"\nPart II: {len(ok)} of {len(checks)} checks passed")
    for lab in bad: print("  not passed:", lab)
else: print("SciPy is not available: install it and re-run this cell.")

### What to look for
* **Exactness.** The helper functions equal SciPy's to rounding; the bootstrap of SciPy gives the same standard error and nearly the same interval as ours, with a difference of the size of the simulation noise.
* **BCa against percentile.** The BCa interval corrects for the bias and the skewness of the bootstrap distribution; for the mean of 100 die rolls the two intervals are close, as they should be where the Gaussian approximation is good.